In [1]:
# !pip install -q langchain langchain-community langchain-groq langchain-pinecone \
#     langchain-huggingface sentence-transformers datasets pinecone-client python-dotenv

In [2]:
# !pip install -q "numpy==1.26.4" --force-reinstall

In [3]:
# !pip install -q --upgrade --force-reinstall numpy scipy

In [4]:
# !pip install -q --upgrade \
#     langchain langchain-community langchain-groq langchain-pinecone langchain-huggingface \
#     sentence-transformers datasets pinecone-client python-dotenv \
#     "numpy>=1.26.4,<2.1" "scipy<1.13"

# Document Question Answering System (RAG)

A Retrieval-Augmented Generation system that answers questions using the
rag-mini-wikipedia dataset, built with LangChain, Pinecone, and Groq.



## Architecture

User Question
│
▼
Embed Query (MiniLM, 384-dim)
│
▼
Pinecone Similarity Search (top-4 chunks, cosine)
│
▼
Prompt Template (context + question, grounding rules)
│
▼
LLM (Groq llama-3.1-8b-instant, swappable to Ollama via config)
│
▼
Grounded Answer

In [5]:
!pip install -q --upgrade \
    langchain langchain-community langchain-groq langchain-pinecone langchain-huggingface \
    langchain-ollama sentence-transformers datasets pinecone-client python-dotenv \
    "numpy>=1.26.4,<2.1" "scipy<1.13"

In [6]:
!pip install -q langchain-ollama

In [7]:
from google.colab import userdata
import os

def load_config() -> dict:
    """
    Loads required API keys from Colab Secrets and fails fast if any
    are missing, rather than letting a later cell crash with an
    unclear error deep inside the pipeline.
    """
    required_keys = ["GROQ_API_KEY", "PINECONE_API_KEY"]
    config = {}

    for key in required_keys:
        value = userdata.get(key)
        if not value:
            raise RuntimeError(
                f"Missing secret: {key}. Add it via the Colab Secrets "
                f"panel (key icon, left sidebar) and enable notebook access."
            )
        config[key] = value
        os.environ[key] = value

    config["LLM_PROVIDER"] = "groq"
    config["EMBEDDING_MODEL"] = "sentence-transformers/all-MiniLM-L6-v2"
    config["PINECONE_INDEX_NAME"] = "rag-qa-system"
    config["PINECONE_REGION"] = "us-east-1"
    config["CHUNK_SIZE"] = 500
    config["CHUNK_OVERLAP"] = 50
    config["TOP_K"] = 4

    return config

CONFIG = load_config()
print("Config loaded. Provider:", CONFIG["LLM_PROVIDER"])

Config loaded. Provider: groq


In [8]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    groq_api_key=CONFIG["GROQ_API_KEY"],
    temperature=0,
)
response = llm.invoke("Reply with exactly: Groq connection OK")
print(response.content)

Groq connection OK


In [9]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=CONFIG["PINECONE_API_KEY"])
print("Pinecone connected. Existing indexes:", pc.list_indexes().names())

Pinecone connected. Existing indexes: ['rag-qa-system']


#Loading HF DataSet

In [10]:
from datasets import load_dataset
from langchain_core.documents import Document

def load_passages() -> list[Document]:
    """
    Loads the rag-mini-wikipedia text corpus and wraps each passage
    as a LangChain Document, preserving its source id as metadata.
    Metadata matters here: it's what lets a retrieved chunk be traced
    back to its origin later (Phase 12+), instead of returning bare text.
    """
    raw = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus", split="passages")

    documents = [
        Document(
            page_content=row["passage"],
            metadata={"source_id": row["id"]},
        )
        for row in raw
    ]
    return documents


def load_eval_qa_pairs() -> list[dict]:
    """
    Loads the ground-truth question/answer pairs. Used only in Phase 18
    (Evaluation) — kept separate from load_passages() because ingestion
    and evaluation are different concerns, even though they come from
    the same HF dataset.
    """
    raw = load_dataset("rag-datasets/rag-mini-wikipedia", "question-answer", split="test")
    return [{"question": row["question"], "answer": row["answer"]} for row in raw]


documents = load_passages()
print(f"Loaded {len(documents)} passages")
print("Sample document:")
print("  content:", documents[0].page_content[:150], "...")
print("  metadata:", documents[0].metadata)

Loaded 3200 passages
Sample document:
  content: Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home t ...
  metadata: {'source_id': 0}


In [11]:
import re

def clean_text(text: str) -> str:
    """
    Normalizes whitespace and strips leading/trailing junk left over
    from Wikipedia markup stripping (e.g. "official name in  ; pron.  ,").
    Deliberately conservative: we don't touch punctuation or casing,
    since aggressive cleaning can remove meaning the embedding model
    and LLM both rely on.
    """
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.;:])", r"\1", text)
    return text.strip()


def clean_documents(documents: list[Document]) -> list[Document]:
    """
    Applies clean_text() to every document's content, preserving metadata.
    Returns new Document objects rather than mutating in place — avoids
    surprising side effects if `documents` is reused elsewhere.
    """
    return [
        Document(page_content=clean_text(doc.page_content), metadata=doc.metadata)
        for doc in documents
    ]


documents = clean_documents(documents)
print("Sample after cleaning:")
print(" ", documents[0].page_content[:150], "...")

Sample after cleaning:
  Uruguay (official full name in; pron., Eastern Republic of Uruguay) is a country located in the southeastern part of South America. It is home to 3.3  ...


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

lengths = [len(doc.page_content) for doc in documents]
print(f"Passage length — min: {min(lengths)}, max: {max(lengths)}, "
      f"avg: {sum(lengths)/len(lengths):.0f}, median: {sorted(lengths)[len(lengths)//2]}")

CHUNK_SIZE = CONFIG["CHUNK_SIZE"]       # 500
CHUNK_OVERLAP = CONFIG["CHUNK_OVERLAP"] # 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],

    length_function=len,
)

chunks = splitter.split_documents(documents)

print(f"\n{len(documents)} passages -> {len(chunks)} chunks")
print(f"Avg chunks per passage: {len(chunks) / len(documents):.2f}")
print("\nSample chunk:")
print(" ", chunks[0].page_content[:200])
print("  metadata:", chunks[0].metadata)

Passage length — min: 1, max: 2504, avg: 388, median: 298

3200 passages -> 4568 chunks
Avg chunks per passage: 1.43

Sample chunk:
  Uruguay (official full name in; pron., Eastern Republic of Uruguay) is a country located in the southeastern part of South America. It is home to 3.3 million people, of which 1.7 million live in the c
  metadata: {'source_id': 0}


In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name=CONFIG["EMBEDDING_MODEL"],
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

sample_vector = embedding_model.embed_query(chunks[0].page_content)
print(f"Embedding dimension: {len(sample_vector)}")
print(f"Sample values: {sample_vector[:5]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimension: 384
Sample values: [0.006985378451645374, -0.061498112976551056, -0.06683707237243652, -0.008285993710160255, 0.04050033539533615]


In [14]:
from pinecone import ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import time

INDEX_NAME = CONFIG["PINECONE_INDEX_NAME"]

def get_or_create_index(pc, index_name: str, dimension: int = 384):
    """
    Creates the Pinecone index if it doesn't already exist. Idempotent —
    safe to re-run without duplicating or erroring on re-execution.
    """
    existing = pc.list_indexes().names()
    if index_name not in existing:
        pc.create_index(
            name=index_name,
            dimension=dimension,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region=CONFIG["PINECONE_REGION"]),
        )
        while not pc.describe_index(index_name).status["ready"]:
            time.sleep(1)
        print(f"Created new index: {index_name}")
    else:
        print(f"Using existing index: {index_name}")

get_or_create_index(pc, INDEX_NAME)

vectorstore = PineconeVectorStore(
    index=pc.Index(INDEX_NAME),
    embedding=embedding_model,
)

vectorstore.add_documents(chunks)
print(f"Upserted {len(chunks)} chunks into '{INDEX_NAME}'")

Using existing index: rag-qa-system
Upserted 4568 chunks into 'rag-qa-system'


In [15]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": CONFIG["TOP_K"]},
)

test_query = "What is Uruguay?"
results = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(results)} chunks:\n")
for i, doc in enumerate(results, 1):
    print(f"[{i}] source_id={doc.metadata.get('source_id')}")
    print(f"    {doc.page_content[:150]}...\n")

Query: What is Uruguay?
Retrieved 4 chunks:

[1] source_id=28.0
    Map of Uruguay...

[2] source_id=28.0
    Map of Uruguay...

[3] source_id=28.0
    Map of Uruguay...

[4] source_id=28.0
    Map of Uruguay...



In [16]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are a helpful assistant answering questions using only the provided context.

Rules:
- Answer using ONLY the information in the context below.
- If the context does not contain enough information to answer, say
  "I don't have enough information in the provided documents to answer that."
- Do not use outside knowledge, even if you know the answer.
- Be concise and direct.

Context:
{context}

Question:
{question}

Answer:"""
)

sample_context = "\n\n".join(doc.page_content for doc in results)
formatted = RAG_PROMPT.format(context=sample_context, question=test_query)
print(formatted[:500], "...")

Human: You are a helpful assistant answering questions using only the provided context.

Rules:
- Answer using ONLY the information in the context below.
- If the context does not contain enough information to answer, say
  "I don't have enough information in the provided documents to answer that."
- Do not use outside knowledge, even if you know the answer.
- Be concise and direct.

Context:
Map of Uruguay

Map of Uruguay

Map of Uruguay

Map of Uruguay

Question:
What is Uruguay?

Answer: ...


In [17]:
from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama

def get_llm(provider: str = None):
    """
    Factory function returning a LangChain chat model, chosen by provider
    name. Downstream code (Phase 15) calls get_llm() and gets back
    something conforming to BaseChatModel — it never branches on which
    provider is active. This is the Strategy Pattern / Dependency
    Inversion Principle in practice.
    """
    provider = provider or CONFIG["LLM_PROVIDER"]

    if provider == "groq":
        return ChatGroq(
            model="llama-3.1-8b-instant",
            groq_api_key=CONFIG["GROQ_API_KEY"],
            temperature=0,
        )
    elif provider == "ollama":
        return ChatOllama(
            model="llama3",
            temperature=0,
        )
    else:
        raise ValueError(
            f"Unknown LLM_PROVIDER: '{provider}'. Expected 'groq' or 'ollama'."
        )

llm = get_llm()
print(f"LLM provider active: {CONFIG['LLM_PROVIDER']}")
print(f"LLM class: {type(llm).__name__}")

LLM provider active: groq
LLM class: ChatGroq


In [18]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs: list) -> str:
    """
    Joins retrieved chunks into a single context string, separated by
    blank lines so the LLM can visually distinguish separate sources.
    """
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

test_question = "What is Uruguay?"
answer = rag_chain.invoke(test_question)
print(f"Q: {test_question}")
print(f"A: {answer}")

Q: What is Uruguay?
A: I don't have enough information in the provided documents to answer that.


In [19]:
def ask(question: str) -> str:
    """
    Single entrypoint for asking the RAG system a question.
    Wraps rag_chain.invoke() with basic error handling so a transient
    API failure doesn't crash the whole notebook session.
    """
    if not question or not question.strip():
        return "Please enter a non-empty question."

    try:
        return rag_chain.invoke(question)
    except Exception as e:
        return f"Error generating answer: {e}"

for q in [
    "What is Uruguay?",
    "Tell me about Abraham Lincoln.",
    "What is the capital of a country that doesn't exist in this dataset?",
]:
    print(f"Q: {q}")
    print(f"A: {ask(q)}\n")

Q: What is Uruguay?
A: I don't have enough information in the provided documents to answer that.

Q: Tell me about Abraham Lincoln.
A: I don't have enough information in the provided documents to answer that.

Q: What is the capital of a country that doesn't exist in this dataset?
A: I don't have enough information in the provided documents to answer that.



In [20]:
# while True:
#     q = input("\nAsk a question (or 'quit'): ")
#     if q.strip().lower() == "quit":
#         break
#     print("\n" + ask(q))

In [21]:
def run_tests():
    results = []

    vec = embedding_model.embed_query("test sentence")
    results.append(("Embedding dimension is 384", len(vec) == 384))

    docs = retriever.invoke("Uruguay")
    results.append(("Retriever returns <= TOP_K docs", len(docs) <= CONFIG["TOP_K"]))
    results.append(("Retriever returns at least 1 doc", len(docs) >= 1))

    results.append(("Retrieved docs have source_id metadata",
                     all("source_id" in d.metadata for d in docs)))

    oversized = [c for c in chunks if len(c.page_content) > CONFIG["CHUNK_SIZE"] + 50]
    results.append(("No chunk wildly exceeds CHUNK_SIZE", len(oversized) == 0))

    try:
        get_llm(provider="not_a_real_provider")
        results.append(("get_llm raises on invalid provider", False))
    except ValueError:
        results.append(("get_llm raises on invalid provider", True))

    results.append(("ask() rejects empty input", "non-empty" in ask("")))

    out_of_domain_answer = ask("What is the plot of a movie that doesn't exist called Zzyx99?")
    results.append(("Out-of-domain question triggers fallback response",
                     "don't have enough information" in out_of_domain_answer.lower()))

    print(f"{'PASS' if all(r[1] for r in results) else 'FAIL'} — {sum(r[1] for r in results)}/{len(results)} tests passed\n")
    for name, passed in results:
        print(f"  [{'✓' if passed else '✗'}] {name}")

run_tests()

PASS — 8/8 tests passed

  [✓] Embedding dimension is 384
  [✓] Retriever returns <= TOP_K docs
  [✓] Retriever returns at least 1 doc
  [✓] Retrieved docs have source_id metadata
  [✓] No chunk wildly exceeds CHUNK_SIZE
  [✓] get_llm raises on invalid provider
  [✓] ask() rejects empty input
  [✓] Out-of-domain question triggers fallback response


In [22]:
import random

eval_pairs = load_eval_qa_pairs()
print(f"Loaded {len(eval_pairs)} ground-truth Q/A pairs")

random.seed(42)
eval_sample = random.sample(eval_pairs, k=20)

def keyword_overlap_score(generated: str, reference: str) -> float:
    """
    Simple, transparent scoring: fraction of reference answer's
    significant words that appear in the generated answer.
    Not semantic similarity — deliberately crude but easy to explain
    and audit by hand, unlike an LLM-as-judge black box.
    """
    ref_words = {w.lower().strip(".,;:") for w in reference.split() if len(w) > 3}
    gen_lower = generated.lower()
    if not ref_words:
        return 0.0
    matched = sum(1 for w in ref_words if w in gen_lower)
    return matched / len(ref_words)


eval_results = []
for pair in eval_sample:
    generated = ask(pair["question"])
    score = keyword_overlap_score(generated, pair["answer"])
    eval_results.append({
        "question": pair["question"],
        "reference": pair["answer"],
        "generated": generated,
        "score": score,
    })

avg_score = sum(r["score"] for r in eval_results) / len(eval_results)
correct_ish = sum(1 for r in eval_results if r["score"] >= 0.5)

print(f"\nEvaluated on {len(eval_sample)} questions")
print(f"Average keyword overlap score: {avg_score:.2f}")
print(f"Questions with score >= 0.5: {correct_ish}/{len(eval_sample)}")

print("\n--- Worst 3 (lowest score) ---")
for r in sorted(eval_results, key=lambda x: x["score"])[:3]:
    print(f"Q: {r['question']}")
    print(f"  Reference: {r['reference']}")
    print(f"  Generated: {r['generated'][:150]}")
    print(f"  Score: {r['score']:.2f}\n")

Loaded 918 ground-truth Q/A pairs

Evaluated on 20 questions
Average keyword overlap score: 0.10
Questions with score >= 0.5: 2/20

--- Worst 3 (lowest score) ---
Q: What is polar bear's skin color?
  Reference: white or cream
  Generated: I don't have enough information in the provided documents to answer that.
  Score: 0.00

Q: Is Canada a member of the OECD?
  Reference: Yes.
  Generated: I don't have enough information in the provided documents to answer that.
  Score: 0.00

Q: Did Abraham Lincoln live in the Frontier?
  Reference: Yes
  Generated: I don't have enough information in the provided documents to answer that.
  Score: 0.00



###Implementation


##sameple questions

1. What is Uruguay?
2. Who was Abraham Lincoln?
3. What is the capital of Uruguay?
4. Where is Uruguay located?
5. When did Uruguay gain independence?
6. Did Lincoln sign the National Banking Act of 1863?
7. Was Abraham Lincoln the first President of the United States?
8. Tell me about Abraham Lincoln.
9. Explain how Uruguay became independent.
10. What is the population of Mars?

In [23]:
while True:
    q = input("\nAsk a question (or 'quit'): ")
    if q.strip().lower() == "quit":
        break
    print("\n" + ask(q))


Ask a question (or 'quit'): What is Uruguay? Who was Abraham Lincoln? What is the capital of Uruguay? Where is Uruguay located? When did Uruguay gain independence? Did Lincoln sign the National Banking Act of 1863? Was Abraham Lincoln the first President of the United States? Tell me about Abraham Lincoln. Explain how Uruguay became independent. What is the population of Mars?

1. What is Uruguay? 
Uruguay is a constitutional democracy.

2. Who was Abraham Lincoln? 
I don't have enough information in the provided documents to answer that.

3. What is the capital of Uruguay? 
I don't have enough information in the provided documents to answer that.

4. Where is Uruguay located? 
I don't have enough information in the provided documents to answer that.

5. When did Uruguay gain independence? 
Uruguay won its independence in 1828.

6. Did Lincoln sign the National Banking Act of 1863? 
I don't have enough information in the provided documents to answer that.

7. Was Abraham Lincoln the f

## Tech Stack
- **Orchestration:** LangChain (LCEL)
- **Embeddings:** sentence-transformers/all-MiniLM-L6-v2 (local, free)
- **Vector DB:** Pinecone (serverless, cosine similarity)
- **LLM:** Groq (llama-3.1-8b-instant) or Ollama (llama3), switchable via config
- **Dataset:** rag-datasets/rag-mini-wikipedia (3200 passages → 4568 chunks)

## Key Design Decisions
- LCEL over RetrievalQA — explicit, inspectable chain composition
- LLM provider abstraction via factory function (Strategy Pattern)
- temperature=0 for deterministic, grounded output
- Explicit "I don't have enough information" fallback to reduce hallucination
- Deterministic tests for pipeline mechanics; property-based tests for LLM grounding behavior

## Evaluation
Sampled 20 questions from the dataset's ground-truth Q/A pairs, scored via
keyword overlap against reference answers (see Section 12 for full results
and failure analysis).

## Known Limitations
- Ollama path implemented but not live-tested (no local daemon in Colab)
- Keyword-overlap eval is a proxy metric, not semantic accuracy